# Estructuras de Datos en Python — Manipulación 🌳 · Nivel Difícil · Respuestas (uso del profesor)

Este es un **trabajo evaluable**. Trabaja sobre estructuras anidadas — listas dentro de diccionarios, diccionarios dentro de listas — que es la forma en que llegan casi todos los datos reales antes de cargarlos a una tabla.

Todo se resuelve **dentro de este notebook** (no hay archivos `.py` en este trabajo). Cada ejercicio te pide escribir una función; las celdas de prueba que siguen a cada una te dicen qué resultado se espera.

Si algo no te sale a la primera, no es señal de que vayas mal: manipular estructuras anidadas es justamente la parte que cuesta, y es la que hace que después `pandas` se sienta cómodo.


## 0. Los datos

Un curso con cuatro estudiantes. Cada estudiante es un diccionario con su nombre (sucio, como llega de un formulario), su código, y una **lista de evaluaciones**; cada evaluación es a su vez un diccionario con materia, nota y peso.

Fíjate en dos cosas: hay notas en `None` (evaluaciones sin registrar todavía) y hay un estudiante sin ninguna nota. Ese tipo de casos son los que rompen el código escrito a la ligera.

In [ ]:
estudiantes = [
    {
        "nombre": "  maría lopez ",
        "codigo": "EST-001",
        "evaluaciones": [
            {"materia": "Python", "nota": 4.2, "peso": 0.3},
            {"materia": "Estadística", "nota": 3.8, "peso": 0.3},
            {"materia": "Proyecto", "nota": None, "peso": 0.4},
        ],
    },
    {
        "nombre": "CARLOS ruiz",
        "codigo": "EST-002",
        "evaluaciones": [
            {"materia": "Python", "nota": 2.9, "peso": 0.3},
            {"materia": "Estadística", "nota": 3.5, "peso": 0.3},
            {"materia": "Proyecto", "nota": 4.5, "peso": 0.4},
        ],
    },
    {
        "nombre": "ana torres",
        "codigo": "EST-003",
        "evaluaciones": [
            {"materia": "Python", "nota": None, "peso": 0.3},
            {"materia": "Estadística", "nota": None, "peso": 0.3},
            {"materia": "Proyecto", "nota": None, "peso": 0.4},
        ],
    },
    {
        "nombre": " josé ramírez ",
        "codigo": "EST-004",
        "evaluaciones": [
            {"materia": "Python", "nota": 5.0, "peso": 0.3},
            {"materia": "Estadística", "nota": 4.1, "peso": 0.3},
            {"materia": "Proyecto", "nota": 3.2, "peso": 0.4},
        ],
    },
]

print(f"{len(estudiantes)} estudiantes cargados")
print(estudiantes[0]["evaluaciones"][0])


## 1. Limpieza de texto

### ✏️ Ejercicio 1 — `limpiar_nombre(nombre)`

Escribe una función que reciba un nombre "sucio" y devuelva el nombre sin espacios sobrantes al inicio/final y en formato título.

Ejemplo: `limpiar_nombre("  maría lopez ")` → `"María Lopez"`

In [ ]:
def limpiar_nombre(nombre):
    return nombre.strip().title()


for estudiante in estudiantes:
    print(repr(estudiante["nombre"]), "->", repr(limpiar_nombre(estudiante["nombre"])))


## 2. Navegar la estructura anidada

Para llegar a una nota hay que bajar dos niveles: del estudiante a su lista de evaluaciones, y de cada evaluación a su nota.

In [ ]:
# Ejemplo: todas las notas del primer estudiante, incluyendo las que faltan
primero = estudiantes[0]
for evaluacion in primero["evaluaciones"]:
    print(evaluacion["materia"], "->", evaluacion["nota"])


### ✏️ Ejercicio 2 — `notas_registradas(estudiante)`

Devuelve la lista de notas de un estudiante, **omitiendo las que estén en `None`**.

Ejemplos:
- `EST-001` → `[4.2, 3.8]`
- `EST-003` → `[]`

In [ ]:
def notas_registradas(estudiante):
    return [e["nota"] for e in estudiante["evaluaciones"] if e["nota"] is not None]


for estudiante in estudiantes:
    print(estudiante["codigo"], "->", notas_registradas(estudiante))


## 3. Promedios con datos faltantes

### ✏️ Ejercicio 3 — `promedio_simple(estudiante)`

Devuelve el promedio de las notas registradas, o `None` si el estudiante no tiene ninguna nota (¡ojo con dividir por cero!).

Ejemplos: `EST-001` → `4.0`, `EST-003` → `None`

In [ ]:
def promedio_simple(estudiante):
    notas = notas_registradas(estudiante)
    if not notas:
        return None
    return sum(notas) / len(notas)


for estudiante in estudiantes:
    print(estudiante["codigo"], "->", promedio_simple(estudiante))


### ✏️ Ejercicio 4 — `promedio_ponderado(estudiante)`

El promedio simple ignora que las evaluaciones **pesan distinto**. Ahora calcula el promedio ponderado usando el campo `peso`, pero con una regla importante:

> Solo entran las evaluaciones que tienen nota, y el resultado se divide por la suma de **esos** pesos (no por 1.0).

Es decir, si a alguien le falta el Proyecto (peso 0.4), su promedio se calcula sobre el 0.6 que sí tiene registrado. Devuelve `None` si no hay ninguna nota.

Ejemplos:
- `EST-001` → `4.0`  (porque `(4.2*0.3 + 3.8*0.3) / 0.6`)
- `EST-002` → `3.72`
- `EST-004` → `4.01`
- `EST-003` → `None`

In [ ]:
def promedio_ponderado(estudiante):
    suma_ponderada = 0.0
    suma_pesos = 0.0
    for evaluacion in estudiante["evaluaciones"]:
        if evaluacion["nota"] is not None:
            suma_ponderada += evaluacion["nota"] * evaluacion["peso"]
            suma_pesos += evaluacion["peso"]

    if suma_pesos == 0:
        return None
    return suma_ponderada / suma_pesos


for estudiante in estudiantes:
    promedio = promedio_ponderado(estudiante)
    if promedio is None:
        print(estudiante["codigo"], "-> sin notas registradas")
    else:
        print(estudiante["codigo"], "->", round(promedio, 2))


## 4. Modificar la estructura

Hasta aquí solo has **leído** los datos. Ahora vas a cambiarlos: registrar una nota que faltaba, metiéndote hasta el diccionario correcto dentro de la lista correcta.

### ✏️ Ejercicio 5 — `registrar_nota(estudiantes, codigo, materia, nota)`

- Busca el estudiante cuyo `codigo` coincida.
- Dentro de sus evaluaciones, busca la que tenga esa `materia` y asígnale la `nota`.
- Si no existe el estudiante, lanza `ValueError(f"No existe el estudiante con código {codigo}")`.
- Si el estudiante existe pero no tiene esa materia, lanza `ValueError(f"El estudiante {codigo} no tiene la materia {materia}")`.

La función no devuelve nada: su trabajo es **modificar** la estructura que recibió.

In [ ]:
def registrar_nota(estudiantes, codigo, materia, nota):
    for estudiante in estudiantes:
        if estudiante["codigo"] == codigo:
            for evaluacion in estudiante["evaluaciones"]:
                if evaluacion["materia"] == materia:
                    evaluacion["nota"] = nota
                    return
            raise ValueError(f"El estudiante {codigo} no tiene la materia {materia}")
    raise ValueError(f"No existe el estudiante con código {codigo}")


print("Antes:", estudiantes[0]["evaluaciones"][2])
registrar_nota(estudiantes, "EST-001", "Proyecto", 4.6)
print("Después:", estudiantes[0]["evaluaciones"][2])

for codigo, materia in [("EST-999", "Python"), ("EST-002", "Bases de Datos")]:
    try:
        registrar_nota(estudiantes, codigo, materia, 3.0)
    except ValueError as e:
        print("Error esperado:", e)


Vuelve a calcular el promedio ponderado de `EST-001`: cambió solo, porque la función modificó la estructura original — no una copia. Antes era `4.0` (calculado sobre el 60% que tenía registrado); ahora que el Proyecto tiene nota, entra el 100%.

In [ ]:
print("Notas registradas:", notas_registradas(estudiantes[0]))
print("Promedio ponderado EST-001:", round(promedio_ponderado(estudiantes[0]), 2))


## 🎯 Reto final — `ranking(estudiantes)`

Devuelve una lista de tuplas `(nombre_limpio, promedio_ponderado)` ordenada de mayor a menor promedio, dejando por fuera a quien no tenga ninguna nota registrada.

Reutiliza `limpiar_nombre` y `promedio_ponderado` — no repitas su lógica. Pista: `sorted(lista, key=..., reverse=True)`.

In [ ]:
def ranking(estudiantes):
    con_promedio = []
    for estudiante in estudiantes:
        promedio = promedio_ponderado(estudiante)
        if promedio is not None:
            con_promedio.append((limpiar_nombre(estudiante["nombre"]), promedio))

    return sorted(con_promedio, key=lambda par: par[1], reverse=True)


for posicion, (nombre, promedio) in enumerate(ranking(estudiantes), start=1):
    print(f"{posicion}. {nombre}: {promedio:.2f}")


Acabas de hacer a mano, sobre estructuras anidadas, lo mismo que en la próxima sesión harás en una línea con `pandas`: filtrar faltantes, promediar, modificar registros y ordenar. La diferencia es que ahora sabes qué está pasando por debajo — y, sobre todo, sabes qué decisiones se toman con los datos que faltan, que es justo lo que una librería hace por ti sin preguntarte.